LangChain Expression Language (LCEL)

LCEL is LangChain's declarative way to compose chains using the pipe (|) operator. It provides:

A consistent interface (invoke, stream, batch)
Easy composition of prompts, models, parsers, and custom functions
Built-in streaming and async support
This notebook covers LCEL patterns from basic to advanced.

In [14]:
!pip install -q langchain langchain-openai langchain-anthropic langchain-google-genai python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # Loads OPENAI_API_KEY, ANTHROPIC_API_KEY, GOOGLE_API_KEY from .env

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# We'll use these models throughout
openai_llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
openai_llm_critique = ChatOpenAI(model="gpt-4o", temperature=0)
# anthropic_llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.7)
# gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [16]:
# Basic LCEL chain: prompt | model | parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

# Chain components together with the pipe operator
chain = prompt | openai_llm | StrOutputParser()

# invoke() takes the prompt's input variables as a dict
result = chain.invoke({"question": "What is LCEL in LangChain?"})
print(result)

In LangChain, LCEL stands for "LangChain Expression Language." It's a formalized language introduced in LangChain that is designed to express chains and workflows in a more structured and standardized way. LCEL allows developers to define the logic of their applications using a high-level language that can be interpreted and executed by LangChain's runtime. This provides a way to build complex, multi-step workflows that can invoke various tools and services, while maintaining readability and modularity in the code. LCEL helps in abstracting away some of the complexity involved in creating these chains, making it easier to manage and maintain LangChain applications.


RunnableLambda — Custom Functions in Chains

Use RunnableLambda to insert any Python function into an LCEL chain.

In [17]:
from langchain_core.runnables import RunnableLambda

# Custom post-processing function
def word_count(text: str) -> str:
    count = len(text.split())
    return f"{text}\n\n--- Word count: {count} ---"

# Custom post-processing function
def char_count(text: str) -> str:
    count = len(text)
    return f"{text}\n\n--- Char count: {count} ---"

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative writer."),
    ("human", "Write a very short poem about {topic}.")
])

# Chain: prompt → model → parse to string → custom function
chain = prompt | openai_llm | StrOutputParser() | RunnableLambda(word_count) | RunnableLambda(char_count)

result = chain.invoke({"topic": "coffee"})
print(result)

In the morning's gentle hush,  
A brew awakens with a rush.  
Amber warmth in porcelain cradled,  
Dreams and drowsiness unbridled.  

Steam rises, whispers of the bean,  
In its depths, life's rhythm keen.  
Sip by sip, the world unfurls,  
Within this cup, the day swirls.  

--- Word count: 44 ---

--- Char count: 300 ---


Chaining Multiple LLM Calls (Sequential Chains)

Feed the output of one LLM call into the next — e.g., generate then critique.

In [19]:
# Step 1: Generate a joke
joke_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian."),
    ("human", "Tell me a short joke about {topic}.")
])

# Step 2: Critique the joke
critique_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedy critic. Be brief."),
    ("human", "Rate this joke and explain why it's funny or not:\n\n{joke}")
])

# You can use anthropic_llm or other llm as critique
# Sequential chain: generate joke → critique it
chain = (
    joke_prompt
    | openai_llm
    | StrOutputParser()
    # Use a lambda to reshape output for the next prompt
    | RunnableLambda(lambda output: {"joke": output})
    | critique_prompt
    | openai_llm_critique
    | StrOutputParser()
)

result = chain.invoke({"topic": "programming"})
print(result)

Rating: 7/10

Explanation: This joke is funny because it plays on the double meaning of "bugs." In programming, "bugs" refer to errors or glitches in the code. The joke cleverly ties this to the common phenomenon where insects (bugs) are attracted to light. By suggesting that programmers prefer dark mode to avoid "attracting bugs," it humorously implies that using light mode could lead to more programming errors. The humor lies in the clever wordplay and the relatable tech culture reference.


In [ ]:
Batch Processing

LCEL chains support .batch() to process multiple inputs efficiently.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following to {language}. Only output the translation."),
    ("human", "{text}")
])

chain = prompt | openai_llm | StrOutputParser()

# Batch multiple inputs at once
results = chain.batch([
    {"text": "Hello, how are you?", "language": "Spanish"},
    {"text": "Hello, how are you?", "language": "French"},
    {"text": "Hello, how are you?", "language": "Japanese"},
])

for r in results:
    print(r)